# Model Building

Develop forecasting models using the prepared weekly SKU-level demand dataset.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

weekly_sales = pd.read_csv(
    "../data/processed/weekly_sales.csv"
)

weekly_sales["week_start"] = pd.to_datetime(
    weekly_sales["week_start"]
)

print("Weekly data shape:", weekly_sales.shape)
print("Unique SKUs:", weekly_sales["sku_id"].nunique())
print(
    "Date range:",
    weekly_sales["week_start"].min(),
    "to",
    weekly_sales["week_start"].max()
)

Weekly data shape: (10400, 4)
Unique SKUs: 100
Date range: 2022-01-03 00:00:00 to 2023-12-25 00:00:00


## Train and Test Split

Split the weekly data by time to avoid using future demand for model training.

In [2]:
train = weekly_sales[
    weekly_sales["week_start"] < "2023-07-01"
].copy()

test = weekly_sales[
    weekly_sales["week_start"] >= "2023-07-01"
].copy()

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Train period:", train["week_start"].min(), "to", train["week_start"].max())
print("Test period:", test["week_start"].min(), "to", test["week_start"].max())

Train shape: (7800, 4)
Test shape: (2600, 4)
Train period: 2022-01-03 00:00:00 to 2023-06-26 00:00:00
Test period: 2023-07-03 00:00:00 to 2023-12-25 00:00:00


## Validation Split

Create a validation period from the end of the training data for model selection.

In [3]:
train_model = train[
    train["week_start"] < "2023-05-01"
].copy()

validation = train[
    train["week_start"] >= "2023-05-01"
].copy()

print("Model training shape:", train_model.shape)
print("Validation shape:", validation.shape)
print(
    "Model training period:",
    train_model["week_start"].min(),
    "to",
    train_model["week_start"].max()
)
print(
    "Validation period:",
    validation["week_start"].min(),
    "to",
    validation["week_start"].max()
)

Model training shape: (6900, 4)
Validation shape: (900, 4)
Model training period: 2022-01-03 00:00:00 to 2023-04-24 00:00:00
Validation period: 2023-05-01 00:00:00 to 2023-06-26 00:00:00


## Modeling Features

Create previous-week and previous-year demand features for the forecasting model.

In [4]:
# Load daily sales data
daily_sales = pd.read_csv(
    "../data/processed/sales_daily.csv",
    parse_dates=["date"]
)

# Create weekly date
daily_sales["week_start"] = (
    daily_sales["date"]
    - pd.to_timedelta(
        daily_sales["date"].dt.dayofweek,
        unit="D"
    )
)

# Weekly promotion features
weekly_promo = (
    daily_sales
    .groupby(["week_start", "sku_id"])
    .agg(
        promo_rate=("promo_flag", "mean"),
        avg_discount_pct=("discount_pct", "mean")
    )
    .reset_index()
)

# Merge promotion features
weekly_sales = weekly_sales.merge(
    weekly_promo,
    on=["week_start", "sku_id"],
    how="left"
)

# Sort chronologically
weekly_sales = weekly_sales.sort_values(
    ["sku_id", "week_start"]
).reset_index(drop=True)

# Lag features
grouped_demand = weekly_sales.groupby("sku_id")["units_sold"]

weekly_sales["lag_1_week"] = grouped_demand.shift(1)
weekly_sales["lag_4_week"] = grouped_demand.shift(4)
weekly_sales["lag_52_week"] = grouped_demand.shift(52)

# Rolling features using previous weeks only
weekly_sales["rolling_mean_4_week"] = (
    grouped_demand
    .shift(1)
    .rolling(4)
    .mean()
    .reset_index(level=0, drop=True)
)

weekly_sales["rolling_std_4_week"] = (
    grouped_demand
    .shift(1)
    .rolling(4)
    .std()
    .reset_index(level=0, drop=True)
)

# Calendar and seasonality features
weekly_sales["month"] = weekly_sales["week_start"].dt.month

weekly_sales["week_of_year"] = (
    weekly_sales["week_start"]
    .dt.isocalendar()
    .week
    .astype(int)
)

weekly_sales["week_sin"] = np.sin(
    2 * np.pi * weekly_sales["week_of_year"] / 52
)

weekly_sales["week_cos"] = np.cos(
    2 * np.pi * weekly_sales["week_of_year"] / 52
)

print("Enhanced feature engineering completed.")
print("Dataset shape:", weekly_sales.shape)

print("\nFeature columns:")
print(weekly_sales.columns.tolist())

Enhanced feature engineering completed.
Dataset shape: (10400, 15)

Feature columns:
['week_start', 'sku_id', 'units_sold', 'seasonal_naive', 'promo_rate', 'avg_discount_pct', 'lag_1_week', 'lag_4_week', 'lag_52_week', 'rolling_mean_4_week', 'rolling_std_4_week', 'month', 'week_of_year', 'week_sin', 'week_cos']


## Training and Validation Features

Create separate feature and target datasets for model training and validation.

In [5]:
model_data = weekly_sales.dropna(
    subset=[
        "lag_1_week",
        "lag_4_week",
        "lag_52_week"
    ]
).copy()

train_model_data = model_data[
    model_data["week_start"] < "2023-05-01"
].copy()

validation_data = model_data[
    (model_data["week_start"] >= "2023-05-01")
    & (model_data["week_start"] <= "2023-06-26")
].copy()

features = [
    "lag_1_week",
    "lag_4_week",
    "lag_52_week",
    "promo_rate",
    "avg_discount_pct",
    "rolling_mean_4_week",
    "rolling_std_4_week",
    "month",
    "week_of_year",
    "week_sin",
    "week_cos"
]
X_train = train_model_data[features]
y_train = train_model_data["units_sold"]

X_validation = validation_data[features]
y_validation = validation_data["units_sold"]

print("Training features:", X_train.shape)
print("Training target:", y_train.shape)
print("Validation features:", X_validation.shape)
print("Validation target:", y_validation.shape)

Training features: (1700, 11)
Training target: (1700,)
Validation features: (900, 11)
Validation target: (900,)


## Random Forest Model

Train a Random Forest regression model using the prepared lag features.

In [6]:
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

print("Random Forest model trained successfully.")

Random Forest model trained successfully.


## Validation Prediction

Generate demand predictions for the validation period using the trained Random Forest model.

In [7]:
validation_data["rf_prediction"] = rf_model.predict(X_validation)

print(
    validation_data[
        ["week_start", "sku_id", "units_sold", "rf_prediction"]
    ].head(10)
)

    week_start      sku_id  units_sold  rf_prediction
69  2023-05-01  S001_P0001         674       1069.835
70  2023-05-08  S001_P0001         697       1030.650
71  2023-05-15  S001_P0001        1204       1025.050
72  2023-05-22  S001_P0001         521        976.135
73  2023-05-29  S001_P0001         686       1015.360
74  2023-06-05  S001_P0001         863       1014.810
75  2023-06-12  S001_P0001        1028       1003.655
76  2023-06-19  S001_P0001         865       1016.540
77  2023-06-26  S001_P0001        1166        989.665
173 2023-05-01  S001_P0002        1059        991.385


## Validation WAPE

Measure the forecasting error of the Random Forest model using WAPE.

In [8]:
actual = validation_data["units_sold"]
forecast = validation_data["rf_prediction"]

rf_wape = (
    (actual - forecast).abs().sum()
    / actual.abs().sum()
) * 100

print(f"Random Forest Validation WAPE: {rf_wape:.2f}%")

Random Forest Validation WAPE: 25.27%


## Baseline Comparison

Compare the Random Forest model with the seasonal-naive baseline on the same validation period.

In [9]:
validation_compare = validation_data.copy()

validation_compare["seasonal_naive"] = weekly_sales.loc[
    validation_compare.index, "seasonal_naive"
].values

baseline_wape = (
    (
        validation_compare["units_sold"]
        - validation_compare["seasonal_naive"]
    ).abs().sum()
    / validation_compare["units_sold"].abs().sum()
) * 100

print(f"Random Forest WAPE: {rf_wape:.2f}%")
print(f"Seasonal Naive WAPE: {baseline_wape:.2f}%")

Random Forest WAPE: 25.27%
Seasonal Naive WAPE: 34.17%


## Rolling-Origin Backtesting

Create multiple historical validation periods to test model performance across different forecast origins.

In [10]:
backtest_data = weekly_sales.copy()

origins = [
    "2023-03-06",
    "2023-04-03",
    "2023-05-01"
]

print("Backtesting origins:")

for origin in origins:
    print(origin)

Backtesting origins:
2023-03-06
2023-04-03
2023-05-01


## Rolling-Origin Backtest

Train the Random Forest model at each forecast origin and measure WAPE on the following four weeks.

In [11]:
backtest_results = []

features = [
    "lag_1_week",
    "lag_4_week",
    "lag_52_week",
    "promo_rate",
    "avg_discount_pct",
    "rolling_mean_4_week",
    "rolling_std_4_week",
    "month",
    "week_of_year",
    "week_sin",
    "week_cos"
]

for origin in origins:

    origin_date = pd.Timestamp(origin)
    forecast_end = origin_date + pd.Timedelta(weeks=12)

    backtest_train = weekly_sales[
        weekly_sales["week_start"] < origin_date
    ].dropna(subset=features).copy()

    backtest_test = weekly_sales[
        (weekly_sales["week_start"] >= origin_date)
        & (weekly_sales["week_start"] < forecast_end)
    ].dropna(subset=features).copy()

    X_bt_train = backtest_train[features]
    y_bt_train = backtest_train["units_sold"]

    X_bt_test = backtest_test[features]
    y_bt_test = backtest_test["units_sold"]

    model = RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_bt_train, y_bt_train)

    predictions = model.predict(X_bt_test)

    wape = (
        np.abs(y_bt_test.values - predictions).sum()
        / np.abs(y_bt_test.values).sum()
    ) * 100

    backtest_results.append({
        "origin": origin_date,
        "forecast_weeks": len(backtest_test["week_start"].unique()),
        "wape": wape
    })

backtest_results = pd.DataFrame(backtest_results)

print(backtest_results)
print(
    f"\nAverage Backtest WAPE: "
    f"{backtest_results['wape'].mean():.2f}%"
)

      origin  forecast_weeks       wape
0 2023-03-06              12  25.033563
1 2023-04-03              12  24.615986
2 2023-05-01              12  24.787420

Average Backtest WAPE: 24.81%


## Random Forest Tuning

Test different Random Forest settings and select the configuration with the lowest validation WAPE.

In [12]:
from sklearn.ensemble import RandomForestRegressor

rf_configs = [
    {"n_estimators": 100, "max_depth": None, "min_samples_leaf": 1},
    {"n_estimators": 200, "max_depth": 10, "min_samples_leaf": 1},
    {"n_estimators": 200, "max_depth": 15, "min_samples_leaf": 2},
    {"n_estimators": 300, "max_depth": 15, "min_samples_leaf": 2}
]

tuning_results = []

for config in rf_configs:

    model = RandomForestRegressor(
        random_state=42,
        n_jobs=-1,
        **config
    )

    model.fit(X_train, y_train)

    predictions = model.predict(X_validation)

    wape = (
        np.abs(y_validation.values - predictions).sum()
        / np.abs(y_validation.values).sum()
    ) * 100

    tuning_results.append({
        **config,
        "wape": wape
    })

tuning_results = pd.DataFrame(tuning_results)

print(tuning_results)

   n_estimators  max_depth  min_samples_leaf       wape
0           100        NaN                 1  25.357120
1           200       10.0                 1  24.812799
2           200       15.0                 2  24.958286
3           300       15.0                 2  24.915730


## Tuned Model

Use the Random Forest configuration that achieved the lowest validation WAPE.

In [13]:
best_config = tuning_results.loc[
    tuning_results["wape"].idxmin()
].to_dict()

print("Best configuration:")
print(best_config)

Best configuration:
{'n_estimators': 200.0, 'max_depth': 10.0, 'min_samples_leaf': 1.0, 'wape': 24.81279937666607}


## Final Tuned Model

Train the selected Random Forest configuration on the complete model training period.

In [14]:
final_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=10,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1
)

final_model.fit(X_train, y_train)

print("Final tuned Random Forest model trained successfully.")

Final tuned Random Forest model trained successfully.


## Final Test Features

Prepare the unseen test period using the same features used during model training.

In [15]:
test_data = weekly_sales[
    (weekly_sales["week_start"] >= "2023-07-03")
    & (weekly_sales["week_start"] <= "2023-12-25")
].copy()

test_data = test_data.dropna(
    subset=[
        "lag_1_week",
        "lag_4_week",
        "lag_52_week"
    ]
).copy()

X_test = test_data[features]
y_test = test_data["units_sold"]

print("Test features:", X_test.shape)
print("Test target:", y_test.shape)
print(
    "Test period:",
    test_data["week_start"].min(),
    "to",
    test_data["week_start"].max()
)

Test features: (2600, 11)
Test target: (2600,)
Test period: 2023-07-03 00:00:00 to 2023-12-25 00:00:00


## Final Test Prediction

Generate demand forecasts for the unseen test period using the final tuned model.

In [16]:
test_data["rf_prediction"] = final_model.predict(X_test)

print(
    test_data[
        ["week_start", "sku_id", "units_sold", "rf_prediction"]
    ].head(10)
)

   week_start      sku_id  units_sold  rf_prediction
78 2023-07-03  S001_P0001         611    1035.821614
79 2023-07-10  S001_P0001         797    1043.382594
80 2023-07-17  S001_P0001        1217    1000.073444
81 2023-07-24  S001_P0001        1470     964.050098
82 2023-07-31  S001_P0001         969     909.744012
83 2023-08-07  S001_P0001        1261     967.333209
84 2023-08-14  S001_P0001         999     957.702465
85 2023-08-21  S001_P0001         981     929.574526
86 2023-08-28  S001_P0001         722     961.571834
87 2023-09-04  S001_P0001         771     949.853350


## Final Test WAPE

Evaluate the final tuned model on the unseen test period using WAPE.

In [17]:
test_actual = test_data["units_sold"]
test_forecast = test_data["rf_prediction"]

final_wape = (
    (test_actual - test_forecast).abs().sum()
    / test_actual.abs().sum()
) * 100

print(f"Final Test WAPE: {final_wape:.2f}%")

Final Test WAPE: 23.90%


## Final Model Comparison

Compare the final Random Forest forecast with the seasonal-naive baseline on the same unseen test period.

In [18]:
test_compare = test_data.copy()

test_compare["seasonal_naive"] = weekly_sales.loc[
    test_compare.index,
    "seasonal_naive"
].values

test_baseline_wape = (
    (
        test_compare["units_sold"]
        - test_compare["seasonal_naive"]
    ).abs().sum()
    / test_compare["units_sold"].abs().sum()
) * 100

print(f"Random Forest Test WAPE: {final_wape:.2f}%")
print(f"Seasonal Naive Test WAPE: {test_baseline_wape:.2f}%")

Random Forest Test WAPE: 23.90%
Seasonal Naive Test WAPE: 33.62%


## Gradient Boosting Model

Train a Gradient Boosting regression model using the prepared lag features.

In [19]:
from sklearn.ensemble import GradientBoostingRegressor

gb_model = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)

gb_model.fit(X_train, y_train)

print("Gradient Boosting model trained successfully.")

Gradient Boosting model trained successfully.


## Gradient Boosting Validation Prediction

Generate predictions for the validation period using the Gradient Boosting model.

In [20]:
validation_data["gb_prediction"] = gb_model.predict(X_validation)

print(
    validation_data[
        ["week_start", "sku_id", "units_sold", "gb_prediction"]
    ].head(10)
)

    week_start      sku_id  units_sold  gb_prediction
69  2023-05-01  S001_P0001         674    1193.243553
70  2023-05-08  S001_P0001         697     993.384444
71  2023-05-15  S001_P0001        1204     969.925213
72  2023-05-22  S001_P0001         521     955.768590
73  2023-05-29  S001_P0001         686     938.994085
74  2023-06-05  S001_P0001         863     959.985503
75  2023-06-12  S001_P0001        1028     947.833457
76  2023-06-19  S001_P0001         865     931.571945
77  2023-06-26  S001_P0001        1166     984.532786
173 2023-05-01  S001_P0002        1059     864.056532


## Gradient Boosting Validation WAPE

Evaluate the Gradient Boosting model on the validation period using WAPE.

In [21]:
gb_actual = validation_data["units_sold"]
gb_forecast = validation_data["gb_prediction"]

gb_wape = (
    (gb_actual - gb_forecast).abs().sum()
    / gb_actual.abs().sum()
) * 100

print(f"Gradient Boosting Validation WAPE: {gb_wape:.2f}%")

Gradient Boosting Validation WAPE: 24.92%


## Gradient Boosting Backtesting

Evaluate the Gradient Boosting model across multiple rolling forecast origins.

In [22]:
gb_backtest_results = []

for origin in origins:

    origin_date = pd.Timestamp(origin)
    forecast_end = origin_date + pd.Timedelta(weeks=12)

    backtest_train = weekly_sales[
        weekly_sales["week_start"] < origin_date
    ].dropna(subset=features).copy()

    backtest_test = weekly_sales[
        (weekly_sales["week_start"] >= origin_date)
        & (weekly_sales["week_start"] < forecast_end)
    ].dropna(subset=features).copy()

    X_bt_train = backtest_train[features]
    y_bt_train = backtest_train["units_sold"]

    X_bt_test = backtest_test[features]
    y_bt_test = backtest_test["units_sold"]

    model = GradientBoostingRegressor(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    )

    model.fit(X_bt_train, y_bt_train)

    predictions = model.predict(X_bt_test)

    wape = (
        np.abs(y_bt_test.values - predictions).sum()
        / np.abs(y_bt_test.values).sum()
    ) * 100

    gb_backtest_results.append({
        "origin": origin_date,
        "forecast_weeks": len(
            backtest_test["week_start"].unique()
        ),
        "wape": wape
    })

gb_backtest_results = pd.DataFrame(gb_backtest_results)

print(gb_backtest_results)
print(
    f"\nAverage Gradient Boosting Backtest WAPE: "
    f"{gb_backtest_results['wape'].mean():.2f}%"
)

      origin  forecast_weeks       wape
0 2023-03-06              12  25.150414
1 2023-04-03              12  25.088870
2 2023-05-01              12  24.607452

Average Gradient Boosting Backtest WAPE: 24.95%


## Gradient Boosting Tuning

Test different Gradient Boosting configurations and measure their validation WAPE.

In [23]:
gb_configs = [
    {"n_estimators": 100, "learning_rate": 0.05, "max_depth": 2},
    {"n_estimators": 200, "learning_rate": 0.05, "max_depth": 3},
    {"n_estimators": 300, "learning_rate": 0.05, "max_depth": 3},
    {"n_estimators": 200, "learning_rate": 0.10, "max_depth": 2}
]

gb_tuning_results = []

for config in gb_configs:

    model = GradientBoostingRegressor(
        random_state=42,
        **config
    )

    model.fit(X_train, y_train)

    predictions = model.predict(X_validation)

    wape = (
        np.abs(y_validation.values - predictions).sum()
        / np.abs(y_validation.values).sum()
    ) * 100

    gb_tuning_results.append({
        **config,
        "wape": wape
    })

gb_tuning_results = pd.DataFrame(gb_tuning_results)

print(gb_tuning_results)

   n_estimators  learning_rate  max_depth       wape
0           100           0.05          2  24.353429
1           200           0.05          3  24.919608
2           300           0.05          3  25.223657
3           200           0.10          2  25.043960


## Tuned Gradient Boosting Backtesting

Evaluate the best Gradient Boosting configuration using rolling-origin backtesting.

In [24]:
gb_tuned_backtest = []

for origin in origins:

    origin_date = pd.Timestamp(origin)
    forecast_end = origin_date + pd.Timedelta(weeks=12)

    backtest_train = weekly_sales[
        weekly_sales["week_start"] < origin_date
    ].dropna(subset=features).copy()

    backtest_test = weekly_sales[
        (weekly_sales["week_start"] >= origin_date)
        & (weekly_sales["week_start"] < forecast_end)
    ].dropna(subset=features).copy()

    model = GradientBoostingRegressor(
        n_estimators=100,
        learning_rate=0.05,
        max_depth=2,
        random_state=42
    )

    model.fit(
        backtest_train[features],
        backtest_train["units_sold"]
    )

    predictions = model.predict(
        backtest_test[features]
    )

    wape = (
        np.abs(
            backtest_test["units_sold"].values
            - predictions
        ).sum()
        / np.abs(
            backtest_test["units_sold"].values
        ).sum()
    ) * 100

    gb_tuned_backtest.append({
        "origin": origin_date,
        "forecast_weeks": len(
            backtest_test["week_start"].unique()
        ),
        "wape": wape
    })

gb_tuned_backtest = pd.DataFrame(
    gb_tuned_backtest
)

print(gb_tuned_backtest)
print(
    f"\nAverage Tuned Gradient Boosting "
    f"Backtest WAPE: "
    f"{gb_tuned_backtest['wape'].mean():.2f}%"
)

      origin  forecast_weeks       wape
0 2023-03-06              12  24.188684
1 2023-04-03              12  24.419673
2 2023-05-01              12  24.024131

Average Tuned Gradient Boosting Backtest WAPE: 24.21%


## Model Comparison

Compare the forecasting models using rolling-origin backtest WAPE.

In [25]:
model_comparison = pd.DataFrame({
    "model": [
        "Seasonal Naive",
        "Random Forest",
        "Tuned Gradient Boosting"
    ],
    "backtest_wape": [
        33.71,
        backtest_results["wape"].mean(),
        gb_tuned_backtest["wape"].mean()
    ]
})

print(model_comparison)

                     model  backtest_wape
0           Seasonal Naive      33.710000
1            Random Forest      24.812323
2  Tuned Gradient Boosting      24.210829


In [26]:
model_comparison.to_csv(
    "../data/processed/model_comparison.csv",
    index=False
)

print("Model comparison saved successfully.")

Model comparison saved successfully.


## Final Gradient Boosting Model

Train the selected Gradient Boosting model on the complete model training data.

In [27]:
final_gb_model = GradientBoostingRegressor(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=2,
    random_state=42
)

final_gb_model.fit(
    X_train,
    y_train
)

print("Final Gradient Boosting model trained successfully.")

Final Gradient Boosting model trained successfully.


## Final Test Prediction

Generate forecasts for the unseen test period using the final Gradient Boosting model.

In [28]:
test_data["gb_prediction"] = final_gb_model.predict(
    X_test
)

print(
    test_data[
        [
            "week_start",
            "sku_id",
            "units_sold",
            "gb_prediction"
        ]
    ].head(10)
)

   week_start      sku_id  units_sold  gb_prediction
78 2023-07-03  S001_P0001         611     982.648914
79 2023-07-10  S001_P0001         797     957.225911
80 2023-07-17  S001_P0001        1217     979.857207
81 2023-07-24  S001_P0001        1470     967.830514
82 2023-07-31  S001_P0001         969     969.655907
83 2023-08-07  S001_P0001        1261     969.655907
84 2023-08-14  S001_P0001         999     959.454128
85 2023-08-21  S001_P0001         981     952.658796
86 2023-08-28  S001_P0001         722     969.655907
87 2023-09-04  S001_P0001         771     972.584938


## Final Gradient Boosting WAPE

Evaluate the final Gradient Boosting model on the unseen test period using WAPE.

In [29]:
gb_test_actual = test_data["units_sold"]
gb_test_forecast = test_data["gb_prediction"]

gb_test_wape = (
    (gb_test_actual - gb_test_forecast).abs().sum()
    / gb_test_actual.abs().sum()
) * 100

print(f"Final Gradient Boosting Test WAPE: {gb_test_wape:.2f}%")

Final Gradient Boosting Test WAPE: 23.75%


## Forecast Uncertainty

Create an 80% prediction interval for the final demand forecast.

In [30]:
validation_residuals = (
    validation_data["units_sold"]
    - validation_data["gb_prediction"]
)

lower_error = validation_residuals.quantile(0.10)
upper_error = validation_residuals.quantile(0.90)

test_data["forecast_lower_80"] = (
    test_data["gb_prediction"] + lower_error
).clip(lower=0)

test_data["forecast_upper_80"] = (
    test_data["gb_prediction"] + upper_error
)

print(
    test_data[
        [
            "week_start",
            "sku_id",
            "gb_prediction",
            "forecast_lower_80",
            "forecast_upper_80"
        ]
    ].head(10)
)

   week_start      sku_id  gb_prediction  forecast_lower_80  forecast_upper_80
78 2023-07-03  S001_P0001     982.648914         592.115923        1342.667476
79 2023-07-10  S001_P0001     957.225911         566.692920        1317.244474
80 2023-07-17  S001_P0001     979.857207         589.324216        1339.875770
81 2023-07-24  S001_P0001     967.830514         577.297523        1327.849077
82 2023-07-31  S001_P0001     969.655907         579.122916        1329.674470
83 2023-08-07  S001_P0001     969.655907         579.122916        1329.674470
84 2023-08-14  S001_P0001     959.454128         568.921137        1319.472691
85 2023-08-21  S001_P0001     952.658796         562.125805        1312.677359
86 2023-08-28  S001_P0001     969.655907         579.122916        1329.674470
87 2023-09-04  S001_P0001     972.584938         582.051947        1332.603501


## Risk Scoring Data

Combine the forecast with current inventory and order information for SKU-level risk analysis.

In [31]:

# Risk Scoring Data
# Create a 12-week SKU-level forecast and merge it with latest inventory

forecast_horizon_weeks = 12

forecast_start = test_data["week_start"].min()

horizon_data = test_data[
    test_data["week_start"]
    < forecast_start + pd.Timedelta(weeks=forecast_horizon_weeks)
].copy()

# Create SKU-level forecast

sku_forecast = (
    horizon_data
    .groupby("sku_id")
    .agg(
        forecast_demand=("gb_prediction", "sum"),
        forecast_lower_80=("forecast_lower_80", "sum"),
        forecast_upper_80=("forecast_upper_80", "sum")
    )
    .reset_index()
)

# Load inventory data

inventory = pd.read_csv(
    "../data/processed/inventory_snapshots.csv"
)

inventory["date"] = pd.to_datetime(
    inventory["date"]
)

# Get latest inventory for each SKU

latest_inventory = (
    inventory
    .sort_values("date")
    .groupby("sku_id")
    .tail(1)
    .reset_index(drop=True)
)

# Merge forecast with latest inventory

risk_data = sku_forecast.merge(
    latest_inventory[
        [
            "sku_id",
            "on_hand_units",
            "on_order_units"
        ]
    ],
    on="sku_id",
    how="left"
)

print("Forecast horizon:", forecast_horizon_weeks, "weeks")
print("Forecast rows:", sku_forecast.shape)
print("Latest inventory rows:", latest_inventory.shape)
print("Risk data shape:", risk_data.shape)

print(risk_data.head(10))

Forecast horizon: 12 weeks
Forecast rows: (100, 4)
Latest inventory rows: (100, 4)
Risk data shape: (100, 6)
       sku_id  forecast_demand  forecast_lower_80  forecast_upper_80  \
0  S001_P0001     11570.681266        6884.285374       15890.904020   
1  S001_P0002     11545.433905        6859.038013       15865.656659   
2  S001_P0003     11554.010835        6867.614943       15874.233590   
3  S001_P0004     11492.491676        6806.095784       15812.714430   
4  S001_P0005     11565.795834        6879.399942       15886.018588   
5  S001_P0006     11535.910215        6849.514323       15856.132969   
6  S001_P0007     11643.375741        6956.979849       15963.598496   
7  S001_P0008     11572.486729        6886.090837       15892.709483   
8  S001_P0009     11426.366302        6739.970410       15746.589056   
9  S001_P0010     11499.871045        6813.475153       15820.093800   

   on_hand_units  on_order_units  
0            223              93  
1            217            

In [32]:
print("Test Data Date Range:")
print(test_data["week_start"].min())
print(test_data["week_start"].max())

print("\nForecast Horizon:")
print("Start:", forecast_start)
print("End:", forecast_start + pd.Timedelta(weeks=12))

print("\nHorizon Data Date Range:")
print(horizon_data["week_start"].min())
print(horizon_data["week_start"].max())

Test Data Date Range:
2023-07-03 00:00:00
2023-12-25 00:00:00

Forecast Horizon:
Start: 2023-07-03 00:00:00
End: 2023-09-25 00:00:00

Horizon Data Date Range:
2023-07-03 00:00:00
2023-09-18 00:00:00


## Latest Inventory

Use the latest available inventory position for each SKU.

In [33]:

# ==========================================
# LATEST INVENTORY SNAPSHOT
# ==========================================

inventory = pd.read_csv(
    "../data/processed/inventory_snapshots.csv"
)

inventory["date"] = pd.to_datetime(
    inventory["date"]
)

# Sort inventory records by date

inventory = inventory.sort_values(
    ["sku_id", "date"]
)

# Select the latest inventory record for each SKU

latest_inventory = (
    inventory
    .groupby("sku_id")
    .tail(1)
    .reset_index(drop=True)
)

# Validate latest inventory

print("Inventory rows:", inventory.shape)
print("Latest inventory rows:", latest_inventory.shape)
print("Unique SKUs:", latest_inventory["sku_id"].nunique())

print(
    latest_inventory[
        [
            "date",
            "sku_id",
            "on_hand_units",
            "on_order_units"
        ]
    ].head(10)
)

Inventory rows: (73100, 4)
Latest inventory rows: (100, 4)
Unique SKUs: 100
        date      sku_id  on_hand_units  on_order_units
0 2024-01-01  S001_P0001            223              93
1 2024-01-01  S001_P0002            217              73
2 2024-01-01  S001_P0003             69             191
3 2024-01-01  S001_P0004            338             152
4 2024-01-01  S001_P0005            471             167
5 2024-01-01  S001_P0006            305             114
6 2024-01-01  S001_P0007            256             184
7 2024-01-01  S001_P0008            315             108
8 2024-01-01  S001_P0009            167             188
9 2024-01-01  S001_P0010            167             107


In [34]:

# RISK FORECAST HORIZON AUDIT

forecast_weeks = (
    test_data["week_start"]
    .dropna()
    .sort_values()
    .drop_duplicates()
)


# Select the latest 12 available forecast weeks
# These are closest to the latest inventory date.

risk_horizon_weeks = forecast_weeks.tail(12)

risk_horizon_data = test_data[
    test_data["week_start"].isin(risk_horizon_weeks)
].copy()

print("Selected risk horizon weeks:", len(risk_horizon_weeks))
print("Selected horizon:", risk_horizon_weeks.tolist())
print("Risk horizon rows:", risk_horizon_data.shape)

print(
    "Latest inventory date:",
    inventory["date"].max()
)

Selected risk horizon weeks: 12
Selected horizon: [Timestamp('2023-10-09 00:00:00'), Timestamp('2023-10-16 00:00:00'), Timestamp('2023-10-23 00:00:00'), Timestamp('2023-10-30 00:00:00'), Timestamp('2023-11-06 00:00:00'), Timestamp('2023-11-13 00:00:00'), Timestamp('2023-11-20 00:00:00'), Timestamp('2023-11-27 00:00:00'), Timestamp('2023-12-04 00:00:00'), Timestamp('2023-12-11 00:00:00'), Timestamp('2023-12-18 00:00:00'), Timestamp('2023-12-25 00:00:00')]
Risk horizon rows: (1200, 19)
Latest inventory date: 2024-01-01 00:00:00


In [35]:
# Select inventory before forecast horizon

forecast_start_date = risk_horizon_data["week_start"].min()

latest_inventory = (
    inventory[inventory["date"] <= forecast_start_date]
    .sort_values("date")
    .groupby("sku_id")
    .tail(1)
    .reset_index(drop=True)
)

print("Inventory cutoff date:", forecast_start_date)
print("Latest inventory used:", latest_inventory["date"].max())

Inventory cutoff date: 2023-10-09 00:00:00
Latest inventory used: 2023-10-09 00:00:00


## Updated Risk Data Merge

The twelve-week forecast is merged with the latest inventory snapshot for each SKU.

This dataset will be used for inventory risk calculations.

In [36]:
# Merge 12-week forecast with latest inventory

risk_data = sku_forecast.merge(
    latest_inventory[
        [
            "sku_id",
            "on_hand_units",
            "on_order_units"
        ]
    ],
    on="sku_id",
    how="left"
)

print("Updated Risk Data Shape:", risk_data.shape)

print(
    risk_data[
        [
            "sku_id",
            "forecast_demand",
            "on_hand_units",
            "on_order_units"
        ]
    ].head(10)
)

Updated Risk Data Shape: (100, 6)
       sku_id  forecast_demand  on_hand_units  on_order_units
0  S001_P0001     11570.681266            417             160
1  S001_P0002     11545.433905            500              30
2  S001_P0003     11554.010835            181              98
3  S001_P0004     11492.491676            237             102
4  S001_P0005     11565.795834            404             122
5  S001_P0006     11535.910215            320              25
6  S001_P0007     11643.375741            499             175
7  S001_P0008     11572.486729            253              76
8  S001_P0009     11426.366302            143              20
9  S001_P0010     11499.871045            180              53


## SKU-Level Forecast

Aggregate the forecast horizon to estimate total expected demand for each SKU.

In [37]:

# Use a 12-week operational planning horizon
# Lead time is not available in the source dataset.
# This assumption will be documented in the project report.

forecast_horizon_weeks = 12

forecast_start = test_data["week_start"].min()

sku_forecast = (
    risk_horizon_data
    .groupby("sku_id")
    .agg(
        forecast_demand=("gb_prediction", "sum"),
        forecast_lower_80=("forecast_lower_80", "sum"),
        forecast_upper_80=("forecast_upper_80", "sum")
    )
    .reset_index()
)

print("Forecast horizon:", len(risk_horizon_weeks), "weeks")
print("SKU forecast rows:", sku_forecast.shape)
print(sku_forecast.head(10))

Forecast horizon: 12 weeks
SKU forecast rows: (100, 4)
       sku_id  forecast_demand  forecast_lower_80  forecast_upper_80
0  S001_P0001     11252.346576        6565.950684       15572.569330
1  S001_P0002     11341.000032        6654.604140       15661.222787
2  S001_P0003     11506.353541        6819.957649       15826.576295
3  S001_P0004     11601.552816        6915.156924       15921.775570
4  S001_P0005     11548.212611        6861.816719       15868.435366
5  S001_P0006     11422.098905        6735.703013       15742.321659
6  S001_P0007     11511.139350        6824.743458       15831.362104
7  S001_P0008     10841.652406        6155.256514       15161.875160
8  S001_P0009     11492.393931        6805.998039       15812.616686
9  S001_P0010     11474.157513        6787.761621       15794.380267


## Inventory and Forecast

Combine forecast demand with the latest inventory position for each SKU.

In [38]:
risk_data = sku_forecast.merge(
    latest_inventory[
        [
            "sku_id",
            "on_hand_units",
            "on_order_units"
        ]
    ],
    on="sku_id",
    how="left"
)

print("Risk data shape:", risk_data.shape)
print(risk_data.head(10))

Risk data shape: (100, 6)
       sku_id  forecast_demand  forecast_lower_80  forecast_upper_80  \
0  S001_P0001     11252.346576        6565.950684       15572.569330   
1  S001_P0002     11341.000032        6654.604140       15661.222787   
2  S001_P0003     11506.353541        6819.957649       15826.576295   
3  S001_P0004     11601.552816        6915.156924       15921.775570   
4  S001_P0005     11548.212611        6861.816719       15868.435366   
5  S001_P0006     11422.098905        6735.703013       15742.321659   
6  S001_P0007     11511.139350        6824.743458       15831.362104   
7  S001_P0008     10841.652406        6155.256514       15161.875160   
8  S001_P0009     11492.393931        6805.998039       15812.616686   
9  S001_P0010     11474.157513        6787.761621       15794.380267   

   on_hand_units  on_order_units  
0            417             160  
1            500              30  
2            181              98  
3            237             102  
4     

## Inventory Coverage

Available inventory is calculated using on-hand and on-order units.

Inventory coverage is evaluated against the twelve-week forecast demand.

In [39]:
# Calculate available inventory

risk_data["available_inventory"] = (
    risk_data["on_hand_units"]
    + risk_data["on_order_units"]
)

# Calculate 12 week inventory coverage ratio

risk_data["inventory_coverage_ratio"] = (
    risk_data["available_inventory"]
    / risk_data["forecast_demand"]
)

print(
    risk_data[
        [
            "sku_id",
            "forecast_demand",
            "available_inventory",
            "inventory_coverage_ratio"
        ]
    ].head(10)
)

       sku_id  forecast_demand  available_inventory  inventory_coverage_ratio
0  S001_P0001     11252.346576                  577                  0.051278
1  S001_P0002     11341.000032                  530                  0.046733
2  S001_P0003     11506.353541                  279                  0.024247
3  S001_P0004     11601.552816                  339                  0.029220
4  S001_P0005     11548.212611                  526                  0.045548
5  S001_P0006     11422.098905                  345                  0.030205
6  S001_P0007     11511.139350                  674                  0.058552
7  S001_P0008     10841.652406                  329                  0.030346
8  S001_P0009     11492.393931                  163                  0.014183
9  S001_P0010     11474.157513                  233                  0.020307


In [40]:
print("Inventory Date Range:")
print(inventory["date"].min())
print(inventory["date"].max())

print("\nLatest Inventory Date Per SKU:")
print(latest_inventory["date"].value_counts())

Inventory Date Range:
2022-01-01 00:00:00
2024-01-01 00:00:00

Latest Inventory Date Per SKU:
date
2023-10-09    100
Name: count, dtype: int64


In [41]:
diagnostic = risk_data[
    [
        "sku_id",
        "forecast_demand",
        "on_hand_units",
        "on_order_units"
    ]
].copy()

diagnostic["weekly_forecast"] = (
    diagnostic["forecast_demand"] / 12
)

diagnostic["total_available"] = (
    diagnostic["on_hand_units"]
    + diagnostic["on_order_units"]
)

diagnostic["coverage_weeks"] = (
    diagnostic["total_available"]
    / diagnostic["weekly_forecast"]
)

print(diagnostic.describe())

print(
    diagnostic[
        [
            "sku_id",
            "forecast_demand",
            "on_hand_units",
            "on_order_units",
            "weekly_forecast",
            "coverage_weeks"
        ]
    ].head(10)
)

       forecast_demand  on_hand_units  on_order_units  weekly_forecast  \
count       100.000000     100.000000      100.000000       100.000000   
mean      11453.403763     265.000000       99.240000       954.450314   
std         112.920442     134.204126       53.541661         9.410037   
min       10841.652406      50.000000       20.000000       903.471034   
25%       11394.084834     136.250000       52.750000       949.507069   
50%       11471.836365     276.000000       87.000000       955.986364   
75%       11512.377956     363.500000      148.250000       959.364830   
max       11723.002047     500.000000      195.000000       976.916837   

       total_available  coverage_weeks  
count       100.000000      100.000000  
mean        364.240000        0.381742  
std         149.092075        0.156351  
min         110.000000        0.115586  
25%         255.000000        0.267965  
50%         352.000000        0.369726  
75%         494.500000        0.521124  
max  

# Calculate weekly forecast demand

In [42]:
risk_data["weekly_forecast"] = (
    risk_data["forecast_demand"] / 12
)

risk_data["coverage_weeks"] = (
    risk_data["available_inventory"]
    / risk_data["weekly_forecast"]
)

print(
    risk_data[
        [
            "sku_id",
            "available_inventory",
            "weekly_forecast",
            "coverage_weeks"
        ]
    ].describe()
)

       available_inventory  weekly_forecast  coverage_weeks
count           100.000000       100.000000      100.000000
mean            364.240000       954.450314        0.381742
std             149.092075         9.410037        0.156351
min             110.000000       903.471034        0.115586
25%             255.000000       949.507069        0.267965
50%             352.000000       955.986364        0.369726
75%             494.500000       959.364830        0.521124
max             693.000000       976.916837        0.721513


## Stockout Risk

Identify SKUs where available inventory is below the forecast demand.

In [43]:
# Calculate stockout shortfall

risk_data["stockout_shortfall_units"] = (
    risk_data["forecast_demand"]
    - risk_data["available_inventory"]
).clip(lower=0)

# Identify stockout risk

risk_data["stockout_risk"] = (
    risk_data["stockout_shortfall_units"] > 0
)

print(
    risk_data[
        [
            "sku_id",
            "forecast_demand",
            "available_inventory",
            "stockout_shortfall_units",
            "stockout_risk"
        ]
    ].head(10)
)

       sku_id  forecast_demand  available_inventory  stockout_shortfall_units  \
0  S001_P0001     11252.346576                  577              10675.346576   
1  S001_P0002     11341.000032                  530              10811.000032   
2  S001_P0003     11506.353541                  279              11227.353541   
3  S001_P0004     11601.552816                  339              11262.552816   
4  S001_P0005     11548.212611                  526              11022.212611   
5  S001_P0006     11422.098905                  345              11077.098905   
6  S001_P0007     11511.139350                  674              10837.139350   
7  S001_P0008     10841.652406                  329              10512.652406   
8  S001_P0009     11492.393931                  163              11329.393931   
9  S001_P0010     11474.157513                  233              11241.157513   

   stockout_risk  
0           True  
1           True  
2           True  
3           True  
4           T

## Overstock Risk

Identify SKUs where available inventory is higher than forecast demand.

In [44]:
# Calculate overstock excess units

risk_data["overstock_excess_units"] = (
    risk_data["available_inventory"]
    - risk_data["forecast_demand"]
).clip(lower=0)

# Identify overstock risk

risk_data["overstock_risk"] = (
    risk_data["overstock_excess_units"] > 0
)

print(
    risk_data[
        [
            "sku_id",
            "forecast_demand",
            "available_inventory",
            "overstock_excess_units",
            "overstock_risk"
        ]
    ].head(10)
)

       sku_id  forecast_demand  available_inventory  overstock_excess_units  \
0  S001_P0001     11252.346576                  577                     0.0   
1  S001_P0002     11341.000032                  530                     0.0   
2  S001_P0003     11506.353541                  279                     0.0   
3  S001_P0004     11601.552816                  339                     0.0   
4  S001_P0005     11548.212611                  526                     0.0   
5  S001_P0006     11422.098905                  345                     0.0   
6  S001_P0007     11511.139350                  674                     0.0   
7  S001_P0008     10841.652406                  329                     0.0   
8  S001_P0009     11492.393931                  163                     0.0   
9  S001_P0010     11474.157513                  233                     0.0   

   overstock_risk  
0           False  
1           False  
2           False  
3           False  
4           False  
5         

## Risk Level

Assign a risk level to each SKU based on stockout and overstock conditions.

In [45]:
def assign_risk(row):
    if row["stockout_risk"] and row["overstock_risk"]:
        return "Watch/Volatile"
    elif row["stockout_risk"]:
        return "Reorder Now"
    elif row["overstock_risk"]:
        return "Markdown/Clear"
    else:
        return "Healthy"


risk_data["risk_level"] = risk_data.apply(
    assign_risk,
    axis=1
)

print(
    risk_data[
        [
            "sku_id",
            "stockout_risk",
            "overstock_risk",
            "risk_level"
        ]
    ].head(10)
)

print("\nRisk Level Summary:")
print(risk_data["risk_level"].value_counts())

       sku_id  stockout_risk  overstock_risk   risk_level
0  S001_P0001           True           False  Reorder Now
1  S001_P0002           True           False  Reorder Now
2  S001_P0003           True           False  Reorder Now
3  S001_P0004           True           False  Reorder Now
4  S001_P0005           True           False  Reorder Now
5  S001_P0006           True           False  Reorder Now
6  S001_P0007           True           False  Reorder Now
7  S001_P0008           True           False  Reorder Now
8  S001_P0009           True           False  Reorder Now
9  S001_P0010           True           False  Reorder Now

Risk Level Summary:
risk_level
Reorder Now    100
Name: count, dtype: int64


## SKU Price

Add the average unit price for each SKU to estimate the rupee value at risk.

In [46]:
sales_daily = pd.read_csv(
    "../data/processed/sales_daily.csv"
)

sku_price = (
    sales_daily
    .groupby("sku_id")
    .agg(
        unit_price=("unit_price", "mean")
    )
    .reset_index()
)

risk_data = risk_data.drop(
    columns=["unit_price"],
    errors="ignore"
)

risk_data = risk_data.merge(
    sku_price,
    on="sku_id",
    how="left"
)

print(
    risk_data[
        [
            "sku_id",
            "unit_price",
            "risk_level"
        ]
    ].head(10)
)

       sku_id  unit_price   risk_level
0  S001_P0001   55.057373  Reorder Now
1  S001_P0002   55.453256  Reorder Now
2  S001_P0003   54.337086  Reorder Now
3  S001_P0004   55.605992  Reorder Now
4  S001_P0005   53.894104  Reorder Now
5  S001_P0006   55.290383  Reorder Now
6  S001_P0007   54.703529  Reorder Now
7  S001_P0008   55.131669  Reorder Now
8  S001_P0009   55.400109  Reorder Now
9  S001_P0010   56.620465  Reorder Now


## Rupee Value at Risk

Estimate the rupee value associated with forecast inventory shortfall.

In [47]:
# Calculate rupee value at risk

risk_data["rupee_value_at_risk"] = (
    risk_data["stockout_shortfall_units"]
    * risk_data["unit_price"]
)

print(
    risk_data[
        [
            "sku_id",
            "risk_level",
            "stockout_shortfall_units",
            "unit_price",
            "rupee_value_at_risk"
        ]
    ].head(10)
)

       sku_id   risk_level  stockout_shortfall_units  unit_price  \
0  S001_P0001  Reorder Now              10675.346576   55.057373   
1  S001_P0002  Reorder Now              10811.000032   55.453256   
2  S001_P0003  Reorder Now              11227.353541   54.337086   
3  S001_P0004  Reorder Now              11262.552816   55.605992   
4  S001_P0005  Reorder Now              11022.212611   53.894104   
5  S001_P0006  Reorder Now              11077.098905   55.290383   
6  S001_P0007  Reorder Now              10837.139350   54.703529   
7  S001_P0008  Reorder Now              10512.652406   55.131669   
8  S001_P0009  Reorder Now              11329.393931   55.400109   
9  S001_P0010  Reorder Now              11241.157513   56.620465   

   rupee_value_at_risk  
0        587756.543255  
1        599505.150394  
2        610061.676950  
3        626265.419425  
4        594032.272421  
5        612457.041392  
6        592829.771162  
7        579580.072183  
8        627649.663672  
9

## Risk Prioritization

Sort SKUs by risk level and rupee value at risk to prioritize operational actions.

In [48]:
risk_priority = {
    "Watch/Volatile": 1,
    "Reorder Now": 2,
    "Markdown/Clear": 3,
    "Healthy": 4
}

risk_data["risk_priority"] = (
    risk_data["risk_level"].map(risk_priority)
)

risk_data = risk_data.sort_values(
    ["risk_priority", "rupee_value_at_risk"],
    ascending=[True, False]
).reset_index(drop=True)

print(
    risk_data[
        [
            "sku_id",
            "risk_level",
            "rupee_value_at_risk"
        ]
    ].head(10)
)

       sku_id   risk_level  rupee_value_at_risk
0  S002_P0011  Reorder Now        648360.649245
1  S004_P0012  Reorder Now        644025.469843
2  S004_P0011  Reorder Now        640918.726514
3  S003_P0007  Reorder Now        640585.170028
4  S001_P0010  Reorder Now        636479.566830
5  S003_P0011  Reorder Now        635053.273986
6  S004_P0002  Reorder Now        633806.109897
7  S004_P0005  Reorder Now        633403.425575
8  S002_P0020  Reorder Now        632718.368095
9  S005_P0004  Reorder Now        632401.005015


## Recommended Action

Assign an operational action to each SKU based on its risk level.

In [49]:
action_map = {
    "Reorder Now": "Reorder inventory",
    "Markdown/Clear": "Markdown or clear stock",
    "Watch/Volatile": "Monitor closely",
    "Healthy": "No immediate action"
}

risk_data["recommended_action"] = (
    risk_data["risk_level"].map(action_map)
)

print(
    risk_data[
        [
            "sku_id",
            "risk_level",
            "recommended_action",
            "rupee_value_at_risk"
        ]
    ].head(10)
)

       sku_id   risk_level recommended_action  rupee_value_at_risk
0  S002_P0011  Reorder Now  Reorder inventory        648360.649245
1  S004_P0012  Reorder Now  Reorder inventory        644025.469843
2  S004_P0011  Reorder Now  Reorder inventory        640918.726514
3  S003_P0007  Reorder Now  Reorder inventory        640585.170028
4  S001_P0010  Reorder Now  Reorder inventory        636479.566830
5  S003_P0011  Reorder Now  Reorder inventory        635053.273986
6  S004_P0002  Reorder Now  Reorder inventory        633806.109897
7  S004_P0005  Reorder Now  Reorder inventory        633403.425575
8  S002_P0020  Reorder Now  Reorder inventory        632718.368095
9  S005_P0004  Reorder Now  Reorder inventory        632401.005015


## Risk Summary

Summarize the number of SKUs and rupee value associated with each risk level.

In [50]:
risk_summary = (
    risk_data
    .groupby("risk_level")
    .agg(
        sku_count=("sku_id", "count"),
        total_value_at_risk=("rupee_value_at_risk", "sum")
    )
    .reset_index()
)

print(risk_summary)

    risk_level  sku_count  total_value_at_risk
0  Reorder Now        100         6.114023e+07


In [51]:
print(risk_data["risk_level"].value_counts())

risk_level
Reorder Now    100
Name: count, dtype: int64


In [52]:
print("Latest Inventory Date:")
print(latest_inventory["date"].max())

print("\nForecast Horizon:")
print(risk_horizon_data["week_start"].min())
print(risk_horizon_data["week_start"].max())

Latest Inventory Date:
2023-10-09 00:00:00

Forecast Horizon:
2023-10-09 00:00:00
2023-12-25 00:00:00


In [53]:
print(
    risk_data[
        [
            "sku_id",
            "forecast_demand",
            "available_inventory",
            "inventory_coverage_ratio",
            "stockout_risk",
            "overstock_risk",
            "risk_level"
        ]
    ].head(10)
)

       sku_id  forecast_demand  available_inventory  inventory_coverage_ratio  \
0  S002_P0011     11444.313015                  257                  0.022457   
1  S004_P0012     11610.293958                  284                  0.024461   
2  S004_P0011     11499.521054                  127                  0.011044   
3  S003_P0007     11450.337219                  147                  0.012838   
4  S001_P0010     11474.157513                  233                  0.020307   
5  S003_P0011     11485.261014                  237                  0.020635   
6  S004_P0002     11529.635418                  270                  0.023418   
7  S004_P0005     11471.144414                  264                  0.023014   
8  S002_P0020     11489.584242                  364                  0.031681   
9  S005_P0004     11487.710208                  127                  0.011055   

   stockout_risk  overstock_risk   risk_level  
0           True           False  Reorder Now  
1           

## Risk Decision Grid Check

Check that each SKU follows the defined stockout and overstock decision grid.

In [54]:
risk_data["grid_check"] = (
    risk_data.apply(assign_risk, axis=1)
    == risk_data["risk_level"]
)

print(
    "Decision grid validation:",
    risk_data["grid_check"].all()
)

print(
    "Invalid risk assignments:",
    (~risk_data["grid_check"]).sum()
)

Decision grid validation: True
Invalid risk assignments: 0


## Save Risk Results

Save the final SKU-level forecast and risk results for use in the dashboard and service.

In [55]:
risk_data.to_csv(
    "../data/processed/sku_risk_results.csv",
    index=False
)

print(
    "Risk results saved:",
    risk_data.shape
)

Risk results saved: (100, 20)


In [56]:
forecast_results = test_data[
    [
        "week_start",
        "sku_id",
        "units_sold",
        "gb_prediction",
        "forecast_lower_80",
        "forecast_upper_80"
    ]
].copy()

forecast_results.to_csv(
    "../data/processed/forecast_results.csv",
    index=False
)

print("Forecast results saved:", forecast_results.shape)

Forecast results saved: (2600, 6)


In [57]:
import joblib

joblib.dump(gb_model, "../data/processed/foresight_model.pkl")

print("Final model saved successfully.")

Final model saved successfully.


## Modeling and Risk Analysis Complete

The demand forecasting models were evaluated using rolling-origin backtesting and WAPE.
The selected model was used to generate forecasts, uncertainty intervals, and SKU-level inventory risk results.

In [58]:
print("Modeling notebook completed successfully.")
print("Final Test WAPE:", f"{gb_test_wape:.2f}%")
print("Risk results:", risk_data.shape)
print("Risk results file: ../data/processed/sku_risk_results.csv")

Modeling notebook completed successfully.
Final Test WAPE: 23.75%
Risk results: (100, 20)
Risk results file: ../data/processed/sku_risk_results.csv


In [59]:
print("Total SKUs:", len(risk_data))

print(
    "Stockout Risk Count:",
    risk_data["stockout_risk"].value_counts()
)

print(
    risk_data[
        [
            "forecast_demand",
            "available_inventory",
            "stockout_shortfall_units"
        ]
    ].describe()
)

Total SKUs: 100
Stockout Risk Count: stockout_risk
True    100
Name: count, dtype: int64
       forecast_demand  available_inventory  stockout_shortfall_units
count       100.000000           100.000000                100.000000
mean      11453.403763           364.240000              11089.163763
std         112.920442           149.092075                191.687404
min       10841.652406           110.000000              10512.652406
25%       11394.084834           255.000000              10956.549783
50%       11471.836365           352.000000              11086.647404
75%       11512.377956           494.500000              11242.413447
max       11723.002047           693.000000              11445.975973
